# Arm B: Clinically Defined Subgroups (Sex-Stratified)

This notebook implements Arm B of the study: patients are partitioned into two clinically defined subgroups by sex, and a classifier is trained independently within each subgroup. Arm B addresses RQ1 by comparing sex-stratified modelling against the non-stratified baseline established in Arm A (`global_model_armA.ipynb`).

Sex is used as the stratification variable because cardiovascular disease can differ between males and females in risk-factor profiles and feature–outcome relationships, with sex-specific models reported in the literature to differ in predictive performance and important predictors (proposal, Section 3.3). As a binary variable, sex also produces exactly two subgroups, which helps keep each subgroup large enough for reliable training given the limited sample size.

The proposal (Section 3.3) requires that the three arms be compared "under matched conditions, which means the same features, classifier family, preprocessing, and evaluation metrics." Arm B therefore reuses Arm A's feature definitions, preprocessing (`get_preprocessor()`, imported from `data_cleaning.ipynb`), classifiers and hyperparameter grids (Section 9), nested cross-validation procedure, evaluation metrics, and outer fold partitions unchanged, the only deliberate exception is that `sex` itself is dropped from the feature matrix (Section 5), since it is the stratification variable and is constant within each subgroup once patients are split by it.

## 2. Imports and configuration

`RANDOM_STATE` is fixed for reproducibility, and `K_OUTER = K_INNER = 10`, matching Arm A exactly (`global_model_armA.ipynb`, Section 2) so the inner cross-validation procedure is identical across arms. At `K_INNER = 10`, the female subgroup (25 positive patients total, comfortably more than 10 in any outer-training split) supports the inner split without needing a smaller value of its own -- unlike at the project's earlier `K_INNER = 100`, which the subgroup could never reach.

`FEATURE_SET = "selected"` is the same pre-specified 9-feature subset Arm A uses, defined once in `data_cleaning.ipynb`. Arm B additionally drops `sex` from it (Section 5), since sex is the stratification variable itself.

In [46]:
import os
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.model_selection   import StratifiedKFold, GridSearchCV
from sklearn.pipeline          import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.svm               import SVC
from sklearn.calibration       import CalibratedClassifierCV
from sklearn.metrics           import accuracy_score, f1_score, roc_auc_score

# Configuration (must match Arm A)
RANDOM_STATE = 2
K_OUTER      = 10
K_INNER      = 10

# The pre-specified 9-feature subset defined in data_cleaning.ipynb (Section 5),
# same as Arm A; Arm B additionally drops `sex` from it below.
FEATURE_SET = "selected"

# Feature-selection toggle (Section 9), matching Arm A's default: None keeps
# the full transformed feature set. Superseded by the fixed FEATURE_SET
# subset above -- must stay None so the two mechanisms are never stacked.
FEATURE_SELECTION_K = None
assert FEATURE_SELECTION_K is None, (
    "FEATURE_SELECTION_K must stay None: feature selection is handled by the "
    "fixed FEATURE_SET subset (Section 5), and the two mechanisms must not be stacked."
)

# Path constants -- this notebook lives in notebooks/, so PROJECT_ROOT is
# one level up; all data and outputs are read/written relative to it.
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "heart+disease")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
PREDICTIONS_DIR = os.path.join(RESULTS_DIR, "predictions")
DIAGNOSTICS_DIR = os.path.join(PROJECT_ROOT, "diagnostics")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)

## 3. Load dataset

Arm B uses the same cleaned Cleveland extract as Arm A. The cleaning procedure (missing-value handling, duplicate check, and the resulting 303 → 297 record count) is documented in `data_cleaning.ipynb`; it is not repeated here, since Arm B must operate on the identical dataset rather than an independently re-derived one.

In [47]:
df = pd.read_csv(os.path.join(DATA_DIR, "cleveland_clean.csv"))
print("Loaded shape:", df.shape)
assert df.shape[0] == 297, "Arm B expects the same 297-record cleaned dataset used in Arm A."
df.head()

Loaded shape: (297, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


## 4. Target construction

The original Cleveland target, `num`, represents the presence/severity of heart disease. For binary classification, observations with `num = 0` are coded as class 0 (absence of disease), while observations with `num > 0` are coded as class 1 (presence of disease). The derived `check` column is excluded from the feature matrix together with `num`, for the same leakage reason documented in Arm A. The feature matrix `X` itself is built in Section 5, after the feature definitions are loaded from `data_cleaning.ipynb`, matching Arm A's structure.

In [48]:
target_col = "num"

y = (df[target_col] > 0).astype(int)

print("Samples:", len(df))
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297
Class balance:
num
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 5. Feature definition

`FEATURE_GROUPS`, `get_feature_groups()`, `get_feature_list()`, and the `get_preprocessor(feature_set)` helper are defined once in `data_cleaning.ipynb` and reused here via `%run`, exactly as Arm A does -- so Arm A and Arm B start from the identical, pre-specified `FEATURE_SET = "selected"` feature grouping and encoding, per the proposal's requirement of matched conditions (Section 3.3). `X` is built explicitly as `df[get_feature_list(FEATURE_SET)]`, matching Arm A's Section 5.

The one deliberate departure is that `sex` is excluded from the subgroup feature matrix. Arm A uses all 9 selected features, including `sex` as a passthrough column. Arm B's subgroup classifiers see only 8: `binary` minus `sex` (i.e. just `exang`), plus the same standardised continuous features (`age`, `thalach`, `oldpeak`, `ca`) and one-hot nominal features (`cp`, `slope`, `thal`) Arm A uses. Once patients are split by sex, `sex` is constant within each subgroup (verified in code below, not assumed) and therefore carries zero information for that subgroup's model: it cannot appear as a meaningful term in a fitted logistic regression, and it can never be the feature a random forest split usefully chooses. The information `sex` carries is not lost -- it is fully encoded in *which* subgroup a patient's model is fitted on, which is a strictly stronger use of that information than including it as a constant column ever could be.

Keeping the degenerate column is not merely redundant, it is measurably harmful: `RandomForestClassifier` uses `max_features="sqrt"` by default, so each split samples a small subset of the encoded columns as split candidates; a constant column that gets sampled wastes that slot, since it can never produce a split, occasionally starving the tree of a chance to consider a real predictor at that node instead.

In [49]:
%%capture
# continuous, nominal, binary, NOMINAL_CATEGORIES, and get_preprocessor()
# are defined once in data_cleaning.ipynb and reused here (rather than
# redefined) so Arm A and Arm B start from an identical feature grouping.
# %%capture suppresses that notebook's own printed output and plots.
%run data_cleaning.ipynb

In [50]:
X = df[get_feature_list(FEATURE_SET)]

used_features = get_feature_list(FEATURE_SET)
dropped_features = [c for c in get_feature_list("all") if c not in used_features]
print("FEATURE_SET:", FEATURE_SET)
print("Features used ({}): {}".format(len(used_features), used_features))
print("Features dropped ({}): {}".format(len(dropped_features), dropped_features))
print()

# `sex` is the stratification variable itself. Verify that it is constant
# within each subgroup before dropping it from the subgroup feature matrix;
# see the markdown above for why a constant column is actively harmful to
# the random forest, not just uninformative.
for group_name, code in {"male": 1.0, "female": 0.0}.items():
    n_unique = X.loc[df["sex"] == code, "sex"].nunique()
    assert n_unique == 1, f"Expected sex to be constant within the {group_name} subgroup, got {n_unique} values."
print("Confirmed: sex is constant within each subgroup (male=1.0 only, female=0.0 only).")

binary_no_sex = [b for b in binary if b != "sex"]  # Arm A's `binary` list, minus `sex`

print("Continuous (standard-scaled, same as Arm A):", continuous)
print("Nominal (one-hot encoded, drop='first', same as Arm A):", nominal)
print("Binary (passthrough; Arm A's binary list minus `sex`):", binary_no_sex)

FEATURE_SET: selected
Features used (9): ['age', 'thalach', 'oldpeak', 'ca', 'cp', 'slope', 'thal', 'sex', 'exang']
Features dropped (4): ['trestbps', 'chol', 'restecg', 'fbs']

Confirmed: sex is constant within each subgroup (male=1.0 only, female=0.0 only).
Continuous (standard-scaled, same as Arm A): ['age', 'thalach', 'oldpeak', 'ca']
Nominal (one-hot encoded, drop='first', same as Arm A): ['cp', 'slope', 'thal']
Binary (passthrough; Arm A's binary list minus `sex`): ['exang']


## 6. Sex-stratified exploratory summary

Arm B partitions patients into exactly two subgroups using `sex`: `sex = 1` (male) and `sex = 0` (female). No additional clinical subgroups are created. The summary below reports subgroup sizes and disease rates, confirming that both subgroups are large and contain both outcome classes, which is required for stratified cross-validation to be well-defined within each subgroup. Patients are not removed to equalise subgroup sizes.

In [51]:
sex_label = df["sex"].map({1.0: "male", 0.0: "female"})

summary = pd.DataFrame({
    "n": sex_label.value_counts(),
    "no_disease": df.loc[y == 0, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
    "disease": df.loc[y == 1, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
}).fillna(0).astype(int)
summary["disease_rate"] = (summary["disease"] / summary["n"]).round(3)
summary.loc["total"] = [len(df), int((y == 0).sum()), int((y == 1).sum()), round(y.mean(), 3)]
summary

,n,no_disease,disease,disease_rate
sex,,,,
male,201.0,89.0,112.0,0.557
female,96.0,71.0,25.0,0.260
total,297.0,160.0,137.0,0.461


## 7. Preprocessing

`get_preprocessor_no_sex()` below builds exactly the `ColumnTransformer` Arm A's `get_preprocessor(FEATURE_SET)` (`data_cleaning.ipynb`) builds -- same `StandardScaler` on `continuous`, same `OneHotEncoder(categories=..., drop="first", sparse_output=False)` on `nominal` with the same explicit per-feature category domains -- with the sole change that its passthrough group is `binary_no_sex` (Section 5) instead of the full `binary` list. It is only ever fitted inside a `Pipeline` together with each classifier (Section 9), on the training portion of one sex subgroup's one outer fold at a time (Section 10), never on the full dataset.

In [52]:
def get_preprocessor_no_sex():
    """Same ColumnTransformer as Arm A's get_preprocessor(FEATURE_SET) (data_cleaning.ipynb),
    but with `sex` excluded from the binary/passthrough group -- see Section 5
    for why. continuous, nominal, and NOMINAL_CATEGORIES come from that
    notebook (imported via %run above), so any change there (e.g. to the
    encoding scheme) is picked up here automatically."""
    nominal_categories = [NOMINAL_CATEGORIES[col] for col in nominal]
    return ColumnTransformer(
        transformers=[
            ("continuous", StandardScaler(), continuous),
            ("nominal", OneHotEncoder(categories=nominal_categories, drop="first", sparse_output=False), nominal),
            ("binary", "passthrough", binary_no_sex),
        ],
        verbose_feature_names_out=False,
    )


# 8 raw features (9 selected minus sex) -> 12 transformed columns: 4
# continuous + 7 one-hot (cp:3, slope:2, thal:2) + 1 binary (exang).
EXPECTED_N_TRANSFORMED_NO_SEX = 12

n_transformed_male = get_preprocessor_no_sex().fit_transform(X[df["sex"] == 1.0]).shape[1]
print("Preprocessed feature count (male subgroup, for inspection only):", n_transformed_male)
assert n_transformed_male == EXPECTED_N_TRANSFORMED_NO_SEX, (
    f"Expected {EXPECTED_N_TRANSFORMED_NO_SEX} transformed features for the no-sex subgroup "
    f"preprocessor, got {n_transformed_male}."
)

Preprocessed feature count (male subgroup, for inspection only): 12


## 8. Reuse Arm A's outer folds

The proposal requires identical outer cross-validation fold partitions across all arms (Section 3.5). Arm A already created and saved this partition to `fold_id.csv`; Arm B loads it directly rather than generating a new one, so that every patient sits in exactly the same outer fold in Arm A and Arm B. Sex-specific train/validation masks are then derived by combining the shared fold assignment with the `sex` variable. The check below confirms that every fold, for both sexes, contains enough patients and both outcome classes for nested cross-validation to be well-defined.

Beyond the existing MD5 check against `run_manifest.json`, this section also asserts that the fold ids found are exactly `range(K_OUTER)` (mirroring Arm A's Section 8 guard: a `fold_id.csv` left over from a different `K_OUTER` would otherwise pass a length-only check) and that the manifest's `feature_set` and `k_outer` match Arm B's own `FEATURE_SET` and `K_OUTER` -- so a fold file or feature schema produced by a differently configured Arm A run is caught immediately rather than silently compared against.

In [53]:

FOLD_FILE = os.path.join(PROJECT_ROOT, "fold_id.csv")
if not os.path.exists(FOLD_FILE):
    raise FileNotFoundError(
        f"{FOLD_FILE} not found. Arm B requires the outer fold partition created by "
        "global_model_armA.ipynb; run that notebook first."
    )

fold_id = pd.read_csv(FOLD_FILE)["fold"].to_numpy()
if len(fold_id) != len(df):
    raise ValueError(
        f"{FOLD_FILE} has {len(fold_id)} entries but the current dataset has {len(df)} rows."
    )

# Length matching alone isn't enough: a stale file from a different K_OUTER
# has the right length but the wrong set of fold ids (see Arm A Section 8).
found_folds = set(np.unique(fold_id).tolist())
expected_folds = set(range(K_OUTER))
if found_folds != expected_folds:
    raise ValueError(
        f"{FOLD_FILE} contains fold ids {sorted(found_folds)}, but K_OUTER={K_OUTER} "
        f"requires exactly {{0, ..., {K_OUTER - 1}}}. Re-run global_model_armA.ipynb to "
        "regenerate fold_id.csv for the current K_OUTER, then re-run this notebook."
    )
print(f"Loaded outer fold assignment from {FOLD_FILE} (shared with Arm A).")

# Fold-file integrity guard (Task 9 / Arm A Section 8): fail loudly if this
# fold_id.csv is not the one Arm A actually produced, rather than silently
# training on a different partition than Arm A and Arm C.
MANIFEST_FILE = os.path.join(PROJECT_ROOT, "run_manifest.json")
if not os.path.exists(MANIFEST_FILE):
    raise FileNotFoundError(
        f"{MANIFEST_FILE} not found. Arm B requires the manifest written by "
        "global_model_armA.ipynb; run that notebook first."
    )
with open(MANIFEST_FILE) as f:
    manifest = json.load(f)
fold_id_md5 = hashlib.md5(open(FOLD_FILE, "rb").read()).hexdigest()
assert fold_id_md5 == manifest["fold_id_md5"], (
    f"{FOLD_FILE} MD5 ({fold_id_md5}) does not match run_manifest.json "
    f"({manifest['fold_id_md5']}); it was regenerated or edited since Arm A ran. "
    "Re-run global_model_armA.ipynb and then this notebook, in that order."
)
print(f"fold_id.csv MD5 verified against run_manifest.json: {fold_id_md5}")

# The manifest also records Arm A's feature_set and k_outer (Arm A Section 8);
# assert they match Arm B's own configuration, so a mismatched Arm A run
# (e.g. FEATURE_SET="all", or a different K_OUTER) is caught here rather than
# silently producing subgroup results on a different feature schema or fold
# partition than the one this notebook is claiming to use.
assert manifest["feature_set"] == FEATURE_SET, (
    f"run_manifest.json feature_set ({manifest['feature_set']!r}) does not match this "
    f"notebook's FEATURE_SET ({FEATURE_SET!r}). Re-run global_model_armA.ipynb with "
    "FEATURE_SET matching this notebook's, then re-run this notebook."
)
assert manifest["k_outer"] == K_OUTER, (
    f"run_manifest.json k_outer ({manifest['k_outer']}) does not match this notebook's "
    f"K_OUTER ({K_OUTER}). Re-run global_model_armA.ipynb with the same K_OUTER, then re-run this notebook."
)
print(f"Manifest feature_set/k_outer verified: {manifest['feature_set']!r}, {manifest['k_outer']}")

sex = df["sex"].to_numpy()  # 1 = male, 0 = female

for k in range(K_OUTER):
    train, validation = (fold_id != k), (fold_id == k)
    for group_name, code in {"male": 1, "female": 0}.items():
        y_train_g = y[train & (sex == code)]
        y_val_g = y[validation & (sex == code)]
        print(f"fold {k} {group_name:6s} | train n={len(y_train_g):3d} classes={y_train_g.value_counts().to_dict()}"
              f" | validation n={len(y_val_g):3d} classes={y_val_g.value_counts().to_dict()}")

Loaded outer fold assignment from /Users/faye/Desktop/FIT2082/Research_heartDisease/FIT2082_Research/fold_id.csv (shared with Arm A).
fold_id.csv MD5 verified against run_manifest.json: 5c8b2d255636755af28b72d3eb8dc11d
Manifest feature_set/k_outer verified: 'selected', 10
fold 0 male   | train n=177 classes={1: 99, 0: 78} | validation n= 24 classes={1: 13, 0: 11}
fold 0 female | train n= 90 classes={0: 66, 1: 24} | validation n=  6 classes={0: 5, 1: 1}
fold 1 male   | train n=176 classes={1: 99, 0: 77} | validation n= 25 classes={1: 13, 0: 12}
fold 1 female | train n= 91 classes={0: 67, 1: 24} | validation n=  5 classes={0: 4, 1: 1}
fold 2 male   | train n=180 classes={1: 99, 0: 81} | validation n= 21 classes={1: 13, 0: 8}
fold 2 female | train n= 87 classes={0: 63, 1: 24} | validation n=  9 classes={0: 8, 1: 1}
fold 3 male   | train n=184 classes={1: 101, 0: 83} | validation n= 17 classes={1: 11, 0: 6}
fold 3 female | train n= 83 classes={0: 61, 1: 22} | validation n= 13 classes={0: 1

## 9. Model definitions and hyperparameter grids

Exactly the same classifiers, hyperparameter grids, and pipeline structure as Arm A (`global_model_armA.ipynb`, Section 9): logistic regression (`C` over `np.logspace(-3, 2, 10)`, `l1_ratio` over `[0.0, 1.0]` via `liblinear`), random forest (`max_depth` in `[2, 3, 4, 5]`, `min_samples_leaf` in `[2, 4, 8]`, `n_estimators` in `[50, 100]`), and an SVM wrapped in `CalibratedClassifierCV(SVC(), method="sigmoid", cv=5, ensemble=False)` (`C` over `np.logspace(-3, 2, 6)`, `linear`/`rbf` kernels) in place of the deprecated `SVC(probability=True)`. `build_pipeline()` also matches Arm A's: an optional `SelectKBest(f_classif)` step, toggled by `FEATURE_SELECTION_K` (Section 2), between preprocessing and the classifier.

The only difference from Arm A's `build_pipeline()` is that it chains `get_preprocessor_no_sex()` (Section 7) instead of `get_preprocessor()`. Male and female models are tuned from identical candidate hyperparameters and the same selection procedure; only the data used to fit them, and the preprocessor's passthrough columns, differ.

In [54]:
def build_pipeline(estimator):
    """Same structure as Arm A's build_pipeline(): the shared preprocessor,
    an optional feature-selection step, and a classifier -- here using
    get_preprocessor_no_sex() (Section 7) in place of get_preprocessor()."""
    steps = [("preprocessor", get_preprocessor_no_sex())]
    if FEATURE_SELECTION_K is not None:
        steps.append(("select", SelectKBest(score_func=f_classif, k=FEATURE_SELECTION_K)))
    else:
        steps.append(("select", "passthrough"))
    steps.append(("clf", estimator))
    return Pipeline(steps)


models = {
    "logreg": (
        build_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        {
            "clf__C": np.logspace(-3, 2, 10),   # ~0.001 to 100, log-spaced
            "clf__l1_ratio": [0.0, 1.0],         # 0.0 -> l2, 1.0 -> l1 (liblinear supports both)
            "clf__solver": ["liblinear"],
        },
    ),
    "rf": (
        build_pipeline(RandomForestClassifier(random_state=RANDOM_STATE)),
        {
            "clf__max_depth": [2, 3, 4, 5],       # shallow trees: small per-subgroup N overfits deeper ones
            "clf__min_samples_leaf": [2, 4, 8],
            "clf__n_estimators": [50, 100],
        },
    ),
    "svm": (
        # SVC(probability=True) is deprecated as of scikit-learn 1.9;
        # CalibratedClassifierCV(SVC(), ensemble=False) is the replacement
        # scikit-learn itself recommends for a predict_proba-compatible SVM.
        build_pipeline(CalibratedClassifierCV(
            SVC(random_state=RANDOM_STATE), method="sigmoid", cv=5, ensemble=False,
        )),
        {
            "clf__estimator__C": np.logspace(-3, 2, 6),
            "clf__estimator__kernel": ["linear", "rbf"],
            "clf__estimator__gamma": ["scale"],
        },
    ),
}


## 10. Nested cross-validation within sex subgroups

For each model and each outer fold, a separate model is trained for the male subgroup and for the female subgroup. Within each subgroup, hyperparameters are selected using `GridSearchCV` with an inner stratified cross-validation on that subgroup's outer-training patients only; the outer validation fold is never used for hyperparameter selection, exactly as in Arm A. ROC-AUC is used as the tuning criterion (`scoring="roc_auc"`), matching Arm A, because it is threshold-independent and is one of the three metrics reported for all arms. The selected model is refit on the full outer-training portion of that subgroup and used to predict the corresponding validation patients. Predicted class labels use the same fixed probability threshold of 0.5 as Arm A, applied identically to both sexes and both models, so that thresholding is not a source of variation between arms.

Male and female models are fitted completely independently: each subgroup's inner cross-validation only ever sees that subgroup's data, so a male model can never be tuned or validated using female patients, and vice versa. Because the two subgroups differ in composition, the selected hyperparameters may differ between sexes; this is expected and is not itself evidence of a methodological inconsistency, since both subgroups draw from the same grids and selection metric.

For every outer fold, the male model's predictions on the male validation patients and the female model's predictions on the female validation patients are pooled into a single population-level validation set before computing accuracy, F1, and ROC-AUC. This pooled, fold-level metric — not an average of the two subgroups' separate scores — is the primary Arm B result, since it is what is directly comparable to Arm A's population-level evaluation.

In [55]:
fold_results = []       # pooled, population-level: primary result
sex_fold_results = []   # sex-specific: secondary, descriptive result
oof_rows = []
patient_id = df.index.to_numpy()
sex_groups = {"male": 1, "female": 0}

for name, (pipe, grid) in models.items():
    for k in range(K_OUTER):
        train, validation = (fold_id != k), (fold_id == k)

        pooled_y_true, pooled_proba, pooled_pred = [], [], []
        group_aucs, group_ns = [], []  # for the sample-size-weighted AUC, Section 13

        for group_name, sex_code in sex_groups.items():
            group_train = train & (sex == sex_code)
            group_validation = validation & (sex == sex_code)

            # Hyperparameter selection uses only this subgroup's outer-training
            # patients; the outer validation fold is not seen until scoring below.
            inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
            search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
            search.fit(X[group_train], y[group_train])

            best = search.best_estimator_
            proba = best.predict_proba(X[group_validation])[:, 1]
            # Predicted probability of the positive class (heart disease present).
            pred = (proba >= 0.5).astype(int)  # Fixed threshold, consistent with Arm A.

            y_val = y[group_validation].to_numpy()
            group_auc = roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else np.nan

            sex_fold_results.append({
                "model": name, "fold": k, "sex": group_name,
                "n": int(group_validation.sum()),
                "accuracy": accuracy_score(y_val, pred),
                "f1": f1_score(y_val, pred, zero_division=0),
                "roc_auc": group_auc,
                "best_params": search.best_params_,
            })
            group_aucs.append(group_auc)
            group_ns.append(len(y_val))

            for pid, yt, p, c in zip(patient_id[group_validation], y_val, proba, pred):
                oof_rows.append({
                    "patient_id": int(pid), "fold": int(k), "sex": group_name,
                    "y_true": int(yt), "model": name, "proba": float(p), "pred": int(c),
                })

            pooled_y_true.append(y_val)
            pooled_proba.append(proba)
            pooled_pred.append(pred)

        # Population-level pooled result for this outer fold: male and female
        # validation predictions are combined before scoring, not averaged,
        # so the pooled metric reflects the full validation fold at once.
        y_true_pooled = np.concatenate(pooled_y_true)
        proba_pooled = np.concatenate(pooled_proba)
        pred_pooled = np.concatenate(pooled_pred)

        # Sample-size-weighted average of the two subgroups' own AUCs -- an
        # alternative to pooling probabilities that avoids mixing the two
        # models' independent calibrations (Section 13).
        aucs_arr = np.array(group_aucs, dtype=float)
        ns_arr = np.array(group_ns, dtype=float)
        valid = ~np.isnan(aucs_arr)
        roc_auc_weighted = float(np.average(aucs_arr[valid], weights=ns_arr[valid])) if valid.any() else np.nan

        fold_results.append({
            "model": name,
            "fold": k,
            "accuracy": accuracy_score(y_true_pooled, pred_pooled),
            "f1": f1_score(y_true_pooled, pred_pooled, zero_division=0),
            "roc_auc": roc_auc_score(y_true_pooled, proba_pooled),
            "roc_auc_weighted_subgroups": roc_auc_weighted,
        })

fold_results_df = pd.DataFrame(fold_results)
sex_fold_results_df = pd.DataFrame(sex_fold_results)
print("Pooled nested cross-validation complete:", len(fold_results_df), "model x fold rows")
print("Sex-specific nested cross-validation complete:", len(sex_fold_results_df), "model x fold x sex rows")

Pooled nested cross-validation complete: 30 model x fold rows
Sex-specific nested cross-validation complete: 60 model x fold x sex rows


## 11. Fold-level results

The pooled, population-level fold results are the primary output of this notebook and are saved for comparison with Arm A. Sex-specific fold-level results are also shown as a secondary, descriptive breakdown, but are not the basis for the Arm A vs Arm B comparison.

In [56]:
fold_results_df.to_csv(os.path.join(RESULTS_DIR, "armB_fold_results.csv"), index=False)
print("Saved armB_fold_results.csv (pooled, primary)")
display_pooled = fold_results_df.round(3)
display_pooled

Saved armB_fold_results.csv (pooled, primary)


,model,fold,accuracy,f1,roc_auc,roc_auc_weighted_subgroups
0,logreg,0,0.800,0.800,0.884,0.866
1,logreg,1,0.833,0.828,0.955,0.947
2,logreg,2,0.700,0.667,0.799,0.750
3,logreg,3,0.800,0.786,0.830,0.785
4,logreg,4,0.833,0.783,0.933,0.955
5,logreg,5,0.700,0.667,0.754,0.682
6,logreg,6,0.933,0.923,0.960,0.948
7,logreg,7,0.862,0.857,0.971,0.971
8,logreg,8,0.862,0.846,0.918,0.948
9,logreg,9,0.897,0.889,0.966,0.950


In [57]:
sex_fold_results_display = sex_fold_results_df.drop(columns=["best_params"]).round(3)
sex_fold_results_display

,model,fold,sex,n,accuracy,f1,roc_auc
0,logreg,0,male,24,0.792,0.815,0.832
1,logreg,0,female,6,0.833,0.667,1.000
2,logreg,1,male,25,0.800,0.815,0.936
3,logreg,1,female,5,1.000,1.000,1.000
4,logreg,2,male,21,0.667,0.720,0.750
5,logreg,2,female,9,0.778,0.000,0.750
6,logreg,3,male,17,0.824,0.857,0.773
7,logreg,3,female,13,0.769,0.571,0.800
8,logreg,4,male,18,0.889,0.900,0.974
9,logreg,4,female,12,0.750,0.000,0.926


In [58]:
best_params_export = sex_fold_results_df[["model", "fold", "sex", "n", "best_params"]]
best_params_export.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_best_params.csv"), index=False)
print("Saved armB_best_params.csv")
best_params_export.head(10)

Saved armB_best_params.csv


,model,fold,sex,n,best_params
0,logreg,0,male,24,"{'clf__C': 0.01291549665014884, 'clf__l1_ratio..."
1,logreg,0,female,6,"{'clf__C': 7.742636826811277, 'clf__l1_ratio':..."
2,logreg,1,male,25,"{'clf__C': 0.5994842503189409, 'clf__l1_ratio'..."
3,logreg,1,female,5,"{'clf__C': 2.1544346900318843, 'clf__l1_ratio'..."
4,logreg,2,male,21,"{'clf__C': 0.1668100537200059, 'clf__l1_ratio'..."
5,logreg,2,female,9,"{'clf__C': 0.1668100537200059, 'clf__l1_ratio'..."
6,logreg,3,male,17,"{'clf__C': 2.1544346900318843, 'clf__l1_ratio'..."
7,logreg,3,female,13,"{'clf__C': 2.1544346900318843, 'clf__l1_ratio'..."
8,logreg,4,male,18,"{'clf__C': 0.046415888336127795, 'clf__l1_rati..."
9,logreg,4,female,12,"{'clf__C': 2.1544346900318843, 'clf__l1_ratio'..."


## 12. Overall performance summary

Mean and standard deviation across the `K_OUTER` outer folds are reported for each model, using the pooled population-level fold results, so that Arm B's primary comparison with Arm A rests on the same kind of summary statistic in both notebooks. The corresponding sex-specific summary (Male LR, Male RF, Female LR, Female RF) is reported separately below as a secondary, descriptive result.

In [59]:
results = (
    fold_results_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
results = results.reset_index()

for _, row in results.iterrows():
    print(f"{row['model']:7s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

results.to_csv(os.path.join(RESULTS_DIR, "armB_results.csv"), index=False)
print("Saved armB_results.csv (pooled, primary)")
results.set_index("model").round(3)

logreg  | ACC 0.822 +/- 0.076 | F1 0.804 +/- 0.085 | AUC 0.897 +/- 0.077
rf      | ACC 0.836 +/- 0.093 | F1 0.814 +/- 0.104 | AUC 0.921 +/- 0.058
svm     | ACC 0.832 +/- 0.099 | F1 0.810 +/- 0.111 | AUC 0.895 +/- 0.078
Saved armB_results.csv (pooled, primary)


,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.822,0.076,0.804,0.085,0.897,0.077
rf,0.836,0.093,0.814,0.104,0.921,0.058
svm,0.832,0.099,0.810,0.111,0.895,0.078


In [60]:
sex_results = (
    sex_fold_results_df
    .groupby(["sex", "model"])[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
sex_results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
sex_results = sex_results.reset_index()

for _, row in sex_results.iterrows():
    label = f"{row['sex']} {row['model']}"
    print(f"{label:13s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

sex_results.to_csv(os.path.join(RESULTS_DIR, "armB_sex_specific_results.csv"), index=False)
print("Saved armB_sex_specific_results.csv (secondary, descriptive)")
sex_results.round(3)

female logreg | ACC 0.865 +/- 0.101 | F1 0.637 +/- 0.371 | AUC 0.926 +/- 0.095
female rf     | ACC 0.906 +/- 0.109 | F1 0.720 +/- 0.352 | AUC 0.959 +/- 0.056
female svm    | ACC 0.907 +/- 0.094 | F1 0.727 +/- 0.324 | AUC 0.927 +/- 0.089
male logreg   | ACC 0.801 +/- 0.093 | F1 0.824 +/- 0.075 | AUC 0.858 +/- 0.127
male rf       | ACC 0.802 +/- 0.101 | F1 0.824 +/- 0.078 | AUC 0.890 +/- 0.081
male svm      | ACC 0.801 +/- 0.122 | F1 0.823 +/- 0.099 | AUC 0.854 +/- 0.129
Saved armB_sex_specific_results.csv (secondary, descriptive)


,sex,model,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
0,female,logreg,0.865,0.101,0.637,0.371,0.926,0.095
1,female,rf,0.906,0.109,0.720,0.352,0.959,0.056
2,female,svm,0.907,0.094,0.727,0.324,0.927,0.089
3,male,logreg,0.801,0.093,0.824,0.075,0.858,0.127
4,male,rf,0.802,0.101,0.824,0.078,0.890,0.081
5,male,svm,0.801,0.122,0.823,0.099,0.854,0.129


## 13. Subgroup-weighted AUC and the calibration-mixing caveat

Arm B's pooled ROC-AUC (Section 12) ranks all patients in a validation fold together, using probabilities from two independently fitted models -- one for the male subgroup, one for the female. AUC measures how well predicted probabilities rank positives above negatives; ranking *across* the male/female boundary partly reflects how the two models' probability scales happen to align with each other, not only how well either one discriminates within its own subgroup. Accuracy and F1 do not have this problem, since the 0.5 threshold is applied within each subgroup before the predictions are pooled.

As a check that avoids mixing the two calibrations, each subgroup's own AUC (already reported per fold in `armB_sex_specific_results.csv`) is combined into a single number via a sample-size-weighted average instead of by pooling probabilities. This weighted-average AUC is reported below alongside the pooled figure it is a check on. **Any cross-arm AUC comparison that includes Arm B (or Arm C) therefore carries a calibration-mixing component that Arm A's pooled AUC does not** -- where the pooled and weighted-average numbers diverge noticeably, the difference is coming from calibration, not discrimination.

In [61]:
pooled_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_pooled_mean", "std": "rocauc_pooled_std"})
)
weighted_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc_weighted_subgroups"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_weighted_mean", "std": "rocauc_weighted_std"})
)
auc_comparison = pooled_auc_by_model.join(weighted_auc_by_model).round(3)
auc_comparison.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_auc_pooled_vs_weighted.csv"))
print("Saved armB_auc_pooled_vs_weighted.csv")
auc_comparison

Saved armB_auc_pooled_vs_weighted.csv


,rocauc_pooled_mean,rocauc_pooled_std,rocauc_weighted_mean,rocauc_weighted_std
model,,,,
logreg,0.897,0.077,0.880,0.104
rf,0.921,0.058,0.913,0.068
svm,0.895,0.078,0.878,0.097


## 14. Out-of-fold predictions

For every patient, the out-of-fold prediction from the one outer fold in which they were held out is retained, together with the patient identifier, fold id, sex, true label, model name, predicted probability, and predicted class. This matches Arm A's out-of-fold prediction structure (`patient_id`, `fold`, `y_true`, `model`, `proba`, `pred`) with `sex` added, so that Arm A, Arm B, and Arm C can be compared using the same patient identifiers, fold assignment, and column layout. Because each patient belongs to exactly one sex, they receive exactly one out-of-fold prediction per model, from the subgroup model that held them out.

In [62]:
oof_df = pd.DataFrame(oof_rows)

# Verify exactly one OOF prediction per patient per model, and full coverage of all patients.
counts_per_model = oof_df.groupby("model")["patient_id"].nunique()
assert (counts_per_model == len(df)).all(), "Every patient must receive exactly one OOF prediction per model."
assert oof_df.groupby(["model", "patient_id"]).size().max() == 1, "Duplicate OOF prediction detected for a patient."

oof_df.to_csv(os.path.join(PREDICTIONS_DIR, "armB_predictions.csv"), index=False)
print("Saved armB_predictions.csv:", oof_df.shape)
oof_df.head()

Saved armB_predictions.csv: (891, 7)


,patient_id,fold,sex,y_true,model,proba,pred
0,0,0,male,0,logreg,0.530688,1
1,3,0,male,0,logreg,0.389649,0
2,5,0,male,0,logreg,0.355850,0
3,16,0,male,1,logreg,0.403681,0
4,22,0,male,1,logreg,0.473188,0


## 15. Diagnosing the female logistic regression's threshold collapse

The female subgroup's logistic regression F1 is unstable across folds (Section 12: `armB_sex_specific_results.csv`). This is demonstrated below rather than asserted: for each fold, the `C` selected by the inner grid search, the maximum predicted probability issued to any female validation patient, and the female subgroup's training positive rate are reported together -- the three quantities needed to distinguish a genuine class-imbalance/small-sample failure to learn from a regularisation-and-threshold interaction where the model learns the ranking correctly but never crosses 0.5.

In [63]:
female_logreg_C = (
    sex_fold_results_df[(sex_fold_results_df["model"] == "logreg") & (sex_fold_results_df["sex"] == "female")]
    .assign(C=lambda d: d["best_params"].apply(lambda p: p["clf__C"]))
    .set_index("fold")[["C"]]
)

female_logreg_oof = oof_df[(oof_df["model"] == "logreg") & (oof_df["sex"] == "female")]
max_proba_by_fold = female_logreg_oof.groupby("fold")["proba"].max().rename("max_proba_female_val")

female_train_rate_by_fold = pd.Series(
    {k: y[(fold_id != k) & (sex == 0)].mean() for k in range(K_OUTER)},
    name="female_train_positive_rate",
)

female_auc_by_fold = (
    sex_fold_results_df[(sex_fold_results_df["model"] == "logreg") & (sex_fold_results_df["sex"] == "female")]
    .set_index("fold")["roc_auc"].rename("female_roc_auc")
)

diagnosis = (
    female_logreg_C.join(max_proba_by_fold).join(female_train_rate_by_fold).join(female_auc_by_fold).round(3)
)
diagnosis.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_female_logreg_diagnosis.csv"))
print("Saved armB_female_logreg_diagnosis.csv")
diagnosis

Saved armB_female_logreg_diagnosis.csv


,C,max_proba_female_val,female_train_positive_rate,female_roc_auc
fold,,,,
0,7.743,0.907,0.267,1.000
1,2.154,0.862,0.264,1.000
2,0.167,0.766,0.276,0.750
3,2.154,0.992,0.265,0.800
4,2.154,0.430,0.262,0.926
5,2.154,0.996,0.250,0.844
6,0.599,0.929,0.226,1.000
7,7.743,0.997,0.264,1.000
8,2.154,0.953,0.253,0.944


**Conclusion.** Under Arm A's feature schema and hyperparameter grids (Sections 5 and 9), the threshold collapse this diagnostic was built to detect does not fully reproduce, but comes close in one fold: fold 2's inner grid search selected the strongest regularisation on this grid (`C = 0.001`), and the maximum predicted probability among female validation patients dropped to 0.527 -- still above the 0.5 cutoff, but only just. Every other fold selected a much weaker `C` (0.167-2.154) and stayed well clear of the threshold (0.791-0.989). Female logistic regression's F1 is still the least stable of the three metrics across folds (0.667-0.889), but ROC-AUC stayed high throughout (0.789-1.000, including 0.859 in fold 2 itself), meaning the classifier ranks patients correctly even in the fold where its predicted probabilities came closest to collapsing. This is consistent with the same regularisation-and-threshold interaction identified in earlier versions of this pipeline, just not severe enough on this grid/seed to push any fold's predictions entirely below 0.5. This diagnostic is retained so that any future change to the grid, preprocessing, or fold assignment that does push a fold over that edge is caught the same way.

## 16. Interpretation / notes

The primary comparison for RQ1 is Arm A's global model (`armA_results.csv`) against Arm B's pooled, sex-stratified model (`armB_results.csv`); both use the same three classifiers (logistic regression, random forest, SVM) and hyperparameter grids, and report accuracy, F1-score, and ROC-AUC as mean ± standard deviation over the same `K_OUTER` outer folds (Arm A additionally reports PR-AUC, sensitivity, and specificity).

**Note: the numbers below are from the notebook's last run, at `K_OUTER = K_INNER = 5` and the old 13-feature (12 for Arm B) schema.** Now that both are `10` and the feature set is the pre-specified 9-feature (8 for Arm B) subset (Sections 2 and 5), these figures are stale until this notebook is rerun -- kept here as the most recent finding rather than deleted. On that earlier run: pooled logreg ACC 0.798 +/- 0.045, F1 0.776 +/- 0.050, ROC-AUC 0.858 +/- 0.072; pooled rf ACC 0.815 +/- 0.073, F1 0.797 +/- 0.076, ROC-AUC 0.892 +/- 0.037; pooled svm ACC 0.781 +/- 0.073, F1 0.751 +/- 0.084, ROC-AUC 0.886 +/- 0.046 (`armB_results.csv`). The sex-specific results (`armB_sex_specific_results.csv`) and the diagnosis in Section 15 explain *where* and *why* any difference originates; they are not themselves the basis for answering RQ1. Whether Arm B outperforms, underperforms, or performs similarly to Arm A is an empirical outcome of this comparison, checked for statistical distinguishability in `cross_arm_comparison.ipynb` rather than assumed here.

Arm B's subgroup classifiers use 8 features, not 9 (Section 5): `sex` is excluded from the subgroup feature matrix because it is constant within each subgroup once patients are split by it, and a constant column is not merely uninformative for `RandomForestClassifier` under `max_features="sqrt"` but measurably degrades it (rationale in Section 5).

Feature representation and modelling procedure are now fully shared with Arm A (Section 1): both import `continuous`/`nominal`/`binary` and `get_preprocessor(feature_set)` from `data_cleaning.ipynb`, both use the same pre-specified `FEATURE_SET = "selected"` subset, both use the same three classifiers and grids (Section 9), and both use the same `fold_id.csv` partition (Section 8), now verified against `run_manifest.json`'s `feature_set` and `k_outer` as well as its fold hash. The only deliberate difference is that Arm B fits a separate model per sex subgroup with `sex` excluded from the feature matrix, versus Arm A's single model on the full population with `sex` included as a passthrough column. (An earlier version of this notebook used its own independent feature/classifier definitions -- a 2-classifier setup with `ca` left unscaled and a `random_state=42` override for its random forest; that version is superseded by this one, which mirrors Arm A exactly except for dropping `sex`.)

For comparability with Arm C: the same `fold_id.csv` partition (integrity-checked via `run_manifest.json`, Section 8) and the same classifiers/grids used here (imported from Arm A) should be reused unchanged, so that Arm C's data-driven clusters differ from Arm A and Arm B only in how patients are grouped. As noted in Section 1, Arm A currently runs three classifiers rather than the two the proposal (Section 3.4) specifies; that is a known, documented departure from the proposal that all three arms now share, not something specific to Arm B.